# Data Engineering and Management Workshop

This notebook is the solution version of a 1-hour workshop for engineering students. It uses a fictional global prosthetics charity as the case study.

You will work through four practical themes:

- Ingestion and pipelines
- Data cleaning and logic checks
- Relational storage and joins
- Cybersecurity and privacy masking

Before you start, run the mock data generation script so that the messy CSV file exists in the same folder as this notebook.

## Step 0: Mock Data Setup

The charity receives irregular intake records from remote clinics. The script in the workspace creates `messy_prosthetics_data.csv` with 15 intentionally messy rows and a small device inventory reference table.

Run the script first, then continue into the notebook exercises.

## Section 1: Ingestion & Pipelines (ETL vs ELT)

ETL means **Extract, Transform, Load**. The data is cleaned and validated before it lands in the destination system. For a medical charity, ETL is often the safer choice because privacy rules, security checks, and validation can happen before sensitive records leave the protected processing zone.

ELT means **Extract, Load, Transform**. Raw data is loaded first, then transformed later inside a scalable platform such as a cloud warehouse or data lakehouse. That approach is useful when a charity wants to centralize large volumes of global analytics data and transform it only when needed.

In practice, a medical charity may use ETL for privacy-critical patient workflows and ELT for large-scale reporting once the data has been safely separated from direct identifiers.

In [ ]:
from pathlib import Path

import pandas as pd

csv_candidates = [
    Path('messy_prosthetics_data.csv'),
    Path('..') / 'messy_prosthetics_data.csv',
    Path.cwd() / 'messy_prosthetics_data.csv',
]

for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Could not find messy_prosthetics_data.csv. Run the mock data script first.')

df = pd.read_csv(csv_path)
df.info()

## Section 2: Data Cleaning & Logic Challenges

Data validation checks whether values make sense before they enter downstream systems. Data type enforcement makes sure a field behaves like the type you expect, such as a number, date, or category.

This matters in data engineering because messy inputs can quietly break joins, analytics, and database loading if they are not standardized first.

In [ ]:
import re

CURRENT_YEAR = 2026

def parse_age_input(value):
    if pd.isna(value):
        return None

    if isinstance(value, (int, float)) and not pd.isna(value):
        numeric_value = int(value)
    else:
        match = re.search(r'\d+', str(value))
        if not match:
            return None
        numeric_value = int(match.group())

    if numeric_value < 100:
        return numeric_value
    if numeric_value > 1900:
        return CURRENT_YEAR - numeric_value
    return numeric_value

df['Age_Years'] = df['Age_Input'].apply(parse_age_input)
df[['Patient_ID', 'Age_Input', 'Age_Years']]

In [ ]:
df['Implant_Date_Clean'] = pd.to_datetime(df['Implant_Date'], errors='coerce', dayfirst=True).dt.strftime('%Y-%m-%d')
df['Amputation_Cause_Clean'] = df['Amputation_Cause'].str.lower().str.strip()

df[['Patient_ID', 'Implant_Date', 'Implant_Date_Clean', 'Amputation_Cause', 'Amputation_Cause_Clean']]

## Section 3: Relational Storage & Big Data

Structured data fits neatly into tables, rows, and columns, which is why SQL databases are strong at enforcing rules and relationships. Unstructured data includes free text such as clinical notes, chat logs, and scanned documents.

A primary key uniquely identifies a row in a table. A foreign key points to a matching row in another table. Joins use those keys to combine related data without duplicating everything into one huge file.

For this charity, patient records can be joined to device inventory data so analysts can see which model was fitted, what it cost, and who manufactured it.

In [ ]:
device_inventory = {
    'M210': {'Model_Name': 'AeroStep Lite', 'Cost_USD': 950, 'Manufacturer': 'Stride MedTech'},
    'M302': {'Model_Name': 'FlexFoot Alpha', 'Cost_USD': 1200, 'Manufacturer': 'Global Limb Works'},
    'M404': {'Model_Name': 'CrestFit Pro', 'Cost_USD': 1850, 'Manufacturer': 'OrthoSphere'},
    'M505': {'Model_Name': 'AdaptMax Field', 'Cost_USD': 2100, 'Manufacturer': 'Humanity Mobility Systems'},
}

device_inventory_df = pd.DataFrame.from_dict(device_inventory, orient='index').reset_index()
device_inventory_df = device_inventory_df.rename(columns={'index': 'Device_Model_ID'})
device_inventory_df = device_inventory_df[['Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

patient_device_df = pd.merge(df, device_inventory_df, on='Device_Model_ID', how='inner')
patient_device_df[['Patient_ID', 'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

## Section 4: Cybersecurity & Cloud Privacy (PII Masking)

PII means Personally Identifiable Information. In healthcare and charity settings, names, contact details, and other identifiers must be handled carefully because laws and policies may require them to be protected, minimized, or removed before sharing data with external systems.

Before patient data is uploaded to public cloud storage, it should be anonymized or pseudonymized so that the files are much less useful to an attacker if exposed. Hashing can help replace names with stable tokens that do not reveal the original text.

In [ ]:
import hashlib

def anonymize_name(value):
    normalized_value = str(value).strip().lower()
    return hashlib.sha256(normalized_value.encode('utf-8')).hexdigest()

df['Full_Name'] = df['Full_Name'].apply(anonymize_name)
df[['Patient_ID', 'Full_Name']]

## Wrap-Up

By the end of the exercises, you should be able to explain why data engineering starts with messy inputs, how to clean and validate those inputs, how relational joins connect business tables, and why privacy controls matter before cloud upload.